## Import

In [3]:
import pandas as pd
import numpy as np
import json
import psycopg2
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, classification_report
import joblib
from dotenv import load_dotenv
import os
import ast
import re

## Récupération des données / POSTES

In [4]:
load_dotenv()

USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = 'localhost'
PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")


#variable qui stock l'url d'accès a notre db / sécurisé grace aux infos au dessus dans le .env
DATABASE_URL = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

try:

    #établie la connexion avec la db, comme un lancement de moteur genre
    engine = create_engine(DATABASE_URL)
    
    #ce que ca va aller chercher dans la db
    query = "SELECT id_poste, id_entreprise, titre, description, skills, annee_experience, salaire_min, salaire_max, type_contrat, remote_policy, langues, latitude, longitude FROM poste;"

    #grace a engine, on peut envoyer ma requete query aller chercher les éléments dans la db avec read_sql , panda permet qeu ce soit lisible en python
    df_postes = pd.read_sql(query, engine)
    
    print("Connexion réussie")
    

    fichier = 'dataset_postes.csv'

    df_postes.to_csv(fichier , index=False, encoding='utf-8')



except Exception as e:
    print(e)


invalid literal for int() with base 10: 'None'


## Récupération des données / USER

In [5]:
try:
    
    engine = create_engine(DATABASE_URL)

    query = """
        SELECT 
            u.id_utilisateur,
            u.nom,
            u.prenom,
            u.age,
            u.role,
            u.latitude_secteur,
            u.longitude_secteur,

            pu.id_profil,
            pu.titre_profession,
            pu.description,
            pu.experiences_annees,
            pu.diplomes,
            pu.hard_skills,
            pu.soft_skills,
            pu.langues,
            pu.jobs_passes,
            pu.candidatures_positives,
            pu.candidatures_refusees

        FROM utilisateur u
        JOIN profil_utilisateur pu ON u.id_utilisateur = pu.id_utilisateur
        WHERE u.role = 'candidat'
    """

    df_combined = pd.read_sql(query, engine)
    
    print("fichier csv créé")

    fichier = 'dataset_combined_users.csv'
    final = df_combined.to_csv(fichier, index=False, encoding='utf-8')

except Exception as e:
    print(f"Erreur : {e}")

Erreur : invalid literal for int() with base 10: 'None'


## Préprocessing des données / Texte pour each user et poste

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

french_stop_words = ['le', 'la', 'les', 'un', 'une', 'des', 'du', 'de', 'et', 'en', 'a', 'à', 'pour', 'nous', 'vous', 'dans', 'sur', 'avec', 'qui', 'que', 'recherche', 'équipe', 'entreprise']

postes = pd.read_csv("dataset_postes.csv", quotechar='"', on_bad_lines='skip') # quand il y a des ' ca évite de fiare des bugs
user = pd.read_csv("dataset_combined_users.csv")

vectorizer = TfidfVectorizer(stop_words=french_stop_words, ngram_range=(1,2)) 


# CLEAN SKILLS CAR JSON LOUCHES

def clean_skills(x):
    try:
        skills_list = ast.literal_eval(x) # transforme en liste python lisible sinon crash 
        return ' '.join([k["name"] for k in skills_list]) # en mode pour chaque élement dans name , genre si ya name:javascript ca prend que javascript car faut pas tt prendre sinon ca va fausser le modele.
    except:
        return ''
    
postes["skills_clean"] = postes['skills'].apply(clean_skills)


# display(postes["skills_clean"])
# display(postes['skills'])




# CLEAN text job

postes["text"] = ((postes["titre"].fillna("") + " ") * 3 + (postes["skills_clean"].fillna("") + " ") * 5 + postes["description"].fillna(""))




# CLEAN profils 

def clean_list(tontext):
    try:
        return ' '.join(ast.literal_eval(tontext))
    except:
        return ""
    
user["hard_skills_clean"] = user["hard_skills"].apply(clean_list) # j'utilsie la fonction sur ces deux la car en vrai cest les plus utiles pour les recommandation mais je verrais si je peux pas appliquer sur d'autres pour améliorer le modele
user["jobs_clean"] = user["jobs_passes"].apply(clean_list)




# CLEAN text user 

user["text"] = ( user["titre_profession"].fillna("") + " " + user["hard_skills_clean"].fillna("") + " " + user["jobs_clean"].fillna("") )

print (postes["text"])
print(user["text"])



0      Senior Back-End Developer Senior Back-End Deve...
1      Ingénieur .net / SQL confirmé(e) - Lead - H/F ...
2      Développeur.se  Full stack Java Développeur.se...
3      Principal Software Engineer Python en startup ...
4      Lead Software Engineer Fullstack (Node & Vue) ...
                             ...                        
240    DevOps AWS - Permanent - Full Remote DevOps AW...
241    Senior Mobile Engineer - Paris Senior Mobile E...
242    Fullstack Engineer - NodeJS Fullstack Engineer...
243    Senior developer Full-Stack Ruby on Rails Seni...
244    Senior Mobile Engineer - Montpellier Senior Mo...
Name: text, Length: 245, dtype: str
0      Développeuse Full Stack Python React PostgreSQ...
1      Data Analyst SQL Power BI Python Excel Tableau...
2      Chargée de Marketing Digital SEO Google Ads Ca...
3      Responsable Logistique SAP Gestion des stocks ...
4      Développeuse Mobile Flutter Dart Swift Firebas...
                             ...                    

## Modèle de prédiction / Vectorization + Nearest Neighbors

In [7]:

postes_vectorize = vectorizer.fit_transform(postes['text'])
user_vectorize = vectorizer.transform(user['text'])

model = NearestNeighbors(n_neighbors=3, metric="cosine")
model.fit(postes_vectorize)
distances, indices = model.kneighbors(user_vectorize)

for i, profile in user.iterrows():
    print(f"\n{profile['titre_profession']}")
    
    for j, idx in enumerate(indices[i]):
        job = postes.iloc[idx] 
        
        print(f"{job['titre']} | score: {1 - distances[i][j]:.2f}")

# print(user_vectorize)
# print(postes_vectorize)


Développeuse Full Stack
Développeur.se  Full stack Java | score: 0.18
Développeur Full Stack Java / React F/H | score: 0.14
Ingénieur / Ingénieure Logiciel Senior - Aix en Provence | score: 0.14

Data Analyst
Senior Microsoft Fabric Engineer (R-18875) | score: 0.25
Ingénieur Data H/F | score: 0.24
Alternance - Data Engineer - H/F | score: 0.11

Chargée de Marketing Digital
Stagiaire Product Manager &amp; Marketing | score: 0.16
Stage 2026 Data Scientist - Marketing and commercial effectiveness(H/F/N) | score: 0.13
Customer Program Manager, Enterprise | score: 0.13

Responsable Logistique
Consultant Expérimenté | ERP/SAP | CDI | H/F | score: 0.24
Chef de Projet IT H/F | score: 0.22
Manager SAP PP (Production Planning) | score: 0.20

Développeuse Mobile
Développeu·r·se mobile Flutter | score: 0.29
Ingénieure/Ingénieur développement Mobile iOS - AIX EN PROVENCE | score: 0.26
Ingénieure/Ingénieur développement Mobile iOS - AIX EN PROVENCE | score: 0.26

Chef de Projet IT
Chef de Projet IT

## Exporter le modèle

In [8]:
try:
    joblib.dump(vectorizer, 'vectorizer.joblib')
    joblib.dump(model, 'model_ia.joblib')
    postes.to_pickle('postes_data.pkl') #recup les datas au cas ou cest recommandés
    print("modele exporter")
except:
    print('ca a crash')

modele exporter
